In [1]:
from probill.agents import TeachableMcpAgent, McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools, SseServerParams
from autogen_agentchat.ui import Console
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json
import os
from probill.utils import create_oai_client, load_yaml_file, create_stdio_server, export_component

config_filepath = "./test.yaml"
config = load_yaml_file(config_filepath)

client = create_oai_client(config["client"])

# Make sure the StdioServerParams has the 'type' field set
server_config = config["StdioServerParams"]
if "type" not in server_config:
    server_config["type"] = "StdioServerParams"


In [2]:
# from probill.utils import export_component
# from autogen_core.models import UserMessage

# # Export agent component and convert to dictionary
# agent_component = export_component(agent)
# print(agent_component)
# # print(agent_component.model_dump())

# # # Export client component and convert to dictionary
# # client_component = export_component(client)
# # print(json.dumps(client_component.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
from probill.tools.mcp import McpsWorkbench, McpsServerParams
from probill.agents import McpHostAgent
from autogen_agentchat.agents import AssistantAgent

mcp_tools = config["McpTools"]
print(json.dumps(mcp_tools, indent=2))

workbench_params = []
for key, value in mcp_tools.items():   
    # Create workbenches based on server type
    if value.get("type") == "StdioServerParams":
        params = StdioServerParams(**value["params"])
    elif value.get("type") == "SseServerParams":
        params = SseServerParams(**value["params"])
    else:
        print(f"Unsupported server type for {key}: {value.get('type')}")
    workbench_params.append(
        McpsServerParams(
            server_id=key,
            server_params=params,
        )         
    )
    print(f"Created workbench params for {key}: {params}")

workbench = McpsWorkbench(workbench_params)
await workbench.list_prompts()


agent = McpHostAgent(
    model_client=client,
    workbench=McpsWorkbench(
        workbench_params
    ),
    name="McpHostAgent",
    reflect_on_tool_use=True,
    model_client_stream=True,
)

{
  "mcp-obsidian": {
    "type": "SseServerParams",
    "params": {
      "url": "http://localhost:9009/sse",
      "headers": {
        "OBSIDIAN_API_KEY": "37d8b00e7e67bd386a2961c37aaf451dc1424ea5d88c37e0c71c7c93eeb04f7c"
      },
      "read_timeout_seconds": 30
    }
  },
  "mcp-nccn-evaluation": {
    "type": "StdioServerParams",
    "params": {
      "command": "mcp-nccn",
      "args": [],
      "env": null,
      "read_timeout_seconds": 30
    }
  }
}
Created workbench params for mcp-obsidian: type='SseServerParams' url='http://localhost:9009/sse' headers={'OBSIDIAN_API_KEY': '37d8b00e7e67bd386a2961c37aaf451dc1424ea5d88c37e0c71c7c93eeb04f7c'} timeout=5 sse_read_timeout=300
Created workbench params for mcp-nccn-evaluation: command='mcp-nccn' args=[] env=None cwd=None encoding='utf-8' encoding_error_handler='strict' type='StdioServerParams' read_timeout_seconds=30.0


Error in sse_reader: peer closed connection without sending complete message body (incomplete chunked read)


In [4]:
print(await workbench.list_tools())

[{'server_id': 'mcp-obsidian', 'name': 'obsidian_list_files_in_dir', 'description': 'Lists all files and directories that exist in a specific Obsidian directory.', 'parameters': {'type': 'object', 'properties': {'dirpath': {'type': 'string', 'description': 'Path to list files from (relative to your vault root). Note that empty directories will not be returned.'}}, 'required': ['dirpath'], 'additionalProperties': False}}, {'server_id': 'mcp-obsidian', 'name': 'obsidian_list_files_in_vault', 'description': 'Lists all files and directories in the root directory of your Obsidian vault.', 'parameters': {'type': 'object', 'properties': {}, 'required': [], 'additionalProperties': False}}, {'server_id': 'mcp-obsidian', 'name': 'obsidian_get_file_contents', 'description': 'Return the content of a single file in your vault.', 'parameters': {'type': 'object', 'properties': {'filepath': {'type': 'string', 'description': 'Path to the relevant file (relative to your vault root).', 'format': 'path'}}

In [4]:
res = await workbench.call_tool(name="load_nccn_page", arguments={"page":23})
for item in res.result:
    print(item)


type='TextResultContent' content='[Image]'
type='ImageResultContent' content=<autogen_core._image.Image object at 0x766741468260>
type='TextResultContent' content='Page 23 loaded successfully'
type='TextResultContent' content='Image size: (2200, 1700)'


In [5]:
res = await Console(agent.run_stream(task="Evaluate patient P002"))

---------- TextMessage (user) ----------
Evaluate patient P002
Evaluate patient P002
---------- ToolCallRequestEvent (McpHostAgent) ----------
[FunctionCall(id='call_36w3sfec', arguments='{"evaluation":{"patient_id":"P002","start_page_id":"BINV-20"}}', name='evaluate_patient')]
Result from workbench:
 type='ToolResult' name='evaluate_patient' result=[TextResultContent(type='TextResultContent', content='{\'Page\': \'{"id":"BINV-20","title":"SYSTEMIC TREATMENT OF RECURRENT UNRESECTABLE (LOCAL OR REGIONAL) OR STAGE IV (M1) DISEASE","page_number":33}\'}'), TextResultContent(type='TextResultContent', content='[Image]'), ImageResultContent(type='ImageResultContent', content=<autogen_core._image.Image object at 0x766740370b00>), TextResultContent(type='TextResultContent', content='{\'DecisionPoint\': \'{"id":"DP-BINV-20-1","question":"Does the patient have bone disease?","answer":"Condition(s) matched."}\'}'), TextResultContent(type='TextResultContent', content='{\'TherapyOption\': \'{"id":"T

In [9]:
from probill.utils import export_component

print(export_component(agent))

{
  "provider": "probill.agents.McpHostAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "An agent that provides assistance with ability to use tools.",
  "label": "McpHostAgent",
  "config": {
    "name": "McpHostAgent",
    "model_client": {
      "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for OpenAI hosted models.",
      "label": "OpenAIChatCompletionClient",
      "config": {
        "frequency_penalty": 0.0,
        "max_tokens": 65536,
        "presence_penalty": 0.0,
        "temperature": 0.0,
        "top_p": 0.4,
        "model": "qwen3:32b",
        "api_key": "**********",
        "max_retries": 65535,
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "r1",
          "structured_output": tru

In [ ]:
from autogen_core import CancellationToken
from autogen_agentchat.base import TaskResult, Response, Handoff as HandoffBase
from autogen_core.model_context import ChatCompletionContext
from autogen_core.models import (
    ChatCompletionClient,
    CreateResult,
    UserMessage
)
from typing import Any, List, Optional
from autogen_core.tools import BaseTool
from autogen_agentchat.messages import (
    ModelClientStreamingChunkEvent,
    TextMessage,
    BaseChatMessage
)
from pydantic import BaseModel

class Test_Content(BaseModel):
    secret: str
    markdown: str    

class Test_Message(BaseChatMessage):
    content: Test_Content

    def to_model_message(self) -> TextMessage:
        return TextMessage(
            content=self.content.markdown+"\n" + self.content.secret,
            secret=self.content.secret,
        )
    def to_text(self) -> str:
        return f"{self.content.markdown}\nSecret: {self.content.secret}"
    
    def to_model_text(self):
        return f"{self.content.markdown}\nSecret: {self.content.secret}"
    
async def test_client(
    message: str,
    model_client: ChatCompletionClient,
    model_client_stream: bool,
    tools: List[BaseTool[Any, Any]],
    handoff_tools: List[BaseTool[Any, Any]],
    agent_name: str,
    cancellation_token: CancellationToken,    
):
    message = UserMessage(content=message, source=agent_name)
    if model_client_stream:
        model_result: Optional[CreateResult] = None
        async for chunk in model_client.create_stream(
            [message], cancellation_token=cancellation_token
        ):
            if isinstance(chunk, CreateResult):
                model_result = chunk
            elif isinstance(chunk, str):
                yield ModelClientStreamingChunkEvent(content=chunk, source=agent_name)
            else:
                raise RuntimeError(f"Invalid chunk type: {type(chunk)}")
        if model_result is None:
            raise RuntimeError("No final model result in streaming mode.")
        if isinstance(model_result.content, str):
            yield Response(
                chat_message=Test_Message(
                    content=Test_Content(
                        markdown=f"** Markdown Result:\n{model_result.content}",
                        secret="little_secret",
                    ),
                    source=agent_name,
                    models_usage=model_result.usage,
                ),
                inner_messages="",
                
            )
    else:
        model_result = await model_client.create(
            [message], cancellation_token=cancellation_token
        )
        yield model_result


In [ ]:
await Console(test_client("hi", client, True, [], [], "test", CancellationToken()), output_stats=True)

In [ ]:
agent_config = {
  "provider": "probill.agents.McpHostAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "McpHostAgent is an agent that can connect to MCP servers and manage tools, resources, and prompts.\n    It acts as a host that can understand and utilize tools provided by MCP servers through either\n    SseMcpToolAdapter or StdioMcpToolAdapter.",
  "label": "PatientEvaluator",
  "config": {
    "name": "patient_evaluator",
    "model_client": {
      "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for OpenAI hosted models.",
      "label": "OpenAIChatCompletionClient",
      "config": {
        "temperature": 0,
        "top_p": 1,
        "model": "qwen3:32b",
        "api_key": "**********",
        "model_info": {
          "vision": False,
          "function_calling": True,
          "json_output": True,
          "family": "unknown",
          "structured_output": True
        },
        "base_url": "http://10.0.40.49:11434/v1"
      },
      "model_client_stream": True
    },
    "description": "Assist with patient evaluation.",
    "server_params": {
      "type": "StdioServerParams",  # Add the type discriminator
      "command": "mcp-nccn",
      "args": [],
      "encoding": "utf-8",
      "encoding_error_handler": "strict",
      "read_timeout_seconds": 30
    },
    "tools": [],
    "system_message": "You are a senior clinical assistant familiar with the NCCN guidelines. Use the provided tools to evaluate the patient. Then, review the results carefully, highlight diagnoses(List All patient data), therapy options, and missing patient data (List the missing data ID and candidate values) in a brief conclusion to support the physician in making clinical decisions, only based on the results, don't make it up. Return in markdown format. Say TERMINATE when you're done.",
    "reflect_on_tool_use": True,
    "model_client_stream": True,
    "model_context": {
      "provider": "autogen_core.model_context.UnboundedChatCompletionContext",
      "component_type": "chat_completion_context",
      "version": 1,
      "component_version": 1,
      "description": "An unbounded chat completion context that keeps a view of the all the messages.",
      "label": "UnboundedChatCompletionContext",
      "config": {}
    },
    "tool_call_summary_format": "{result}",
    "metadata": {}
  }
}

In [ ]:
from autogen_agentchat.ui import Console, UserInputManager
from autogen_agentchat.messages import UserMessage, BaseChatMessage,TextMessage
from autogen_agentchat.agents import AssistantAgent, BaseChatAgent
from probill.agents import TeachableMcpAgent, McpHostAgent, McpHostAgentConfig
from probill.utils import check_and_create_server_params
from autogen_core import ComponentLoader
from autogen_core.tools import Workbench
from autogen_ext.tools.mcp import McpWorkbench, SseServerParams
from probill.utils import check_and_create_server_params

msg = TextMessage(content="Generate a complex sample medical report with great details for a TNBC patient, return on a JSON object only.", source="Phil")

# Fix the server_params in the configuration before loading
agent_config["config"]["server_params"]["type"] = "StdioServerParams"

workbench = McpWorkbench(check_and_create_server_params(agent_config["config"]["server_params"]))
await workbench.list_tools()
agent = AssistantAgent(
    model_client=client,
    model_client_stream=True,
    workbench=workbench,
    handoffs=[],
    name="test",
    reflect_on_tool_use=True,
)
# # Now load the component
mcp_host_agent = BaseChatAgent.load_component(agent_config)
mcp_host_agent._model_client_stream = True

In [ ]:
msg = "Please list all the tools that you can use."
res = await Console(agent.run_stream(task=msg))

In [ ]:
print(msg)

In [ ]:
res = await Console(mcp_host_agent.run_stream(task= "Evaluate patient P002"))

In [ ]:
# Alternative way to create an McpHostAgent with proper server parameter validation
from probill.utils import check_and_create_server_params

# Example 1: Create with direct config dict that includes the type field
server_params_dict = {
    "type": "StdioServerParams",  # Type field is required by Pydantic for validation
    "command": "mcp-nccn",
    "args": [],
    "encoding": "utf-8",
    "encoding_error_handler": "strict",
    "read_timeout_seconds": 30
}

# Example 2: Use the check_and_create_server_params utility to ensure proper type
validated_server_params = check_and_create_server_params(server_params_dict)
print(f"Validated server params type: {type(validated_server_params)}")

# Create an agent with the validated server params
mcp_host_agent_from_validated = McpHostAgent(
    name="mcp_agent",
    model_client=client,
    description="Agent using validated server params",
    server_params=validated_server_params, 
    model_client_stream=True,
    system_message="You are a helpful clinical assistant.",
    reflect_on_tool_use=True,
)

# Example 3: Create from agent config using proper config schema
mcp_agent_config = McpHostAgentConfig(
    name="config_agent",
    model_client=client,
    description="Agent created from explicit config",
    server_params=validated_server_params,
    model_client_stream=True,
    system_message="You are a helpful clinical assistant.",
    reflect_on_tool_use=True,
)

# Create agent from config
mcp_host_agent_from_config = McpHostAgent._from_config(mcp_agent_config)
print("Successfully created McpHostAgent from config!")

In [ ]:
from probill.utils import check_and_create_server_params
from autogen_ext.tools.mcp import mcp_server_tools

# Extract server_params from the config and ensure it has the type field
server_params = agent_config["config"]["server_params"]
tools = await mcp_server_tools(check_and_create_server_params(server_params))

In [ ]:
# Let's try running the mcp_host_agent that we loaded
try:
    res = await mcp_host_agent.run(task="Hi")
    print("Successfully executed the agent task")
except Exception as e:
    print(f"Error executing the agent: {e}")
    # Let's see what type of agent we have
    print(f"Agent type: {type(mcp_host_agent)}")

In [ ]:
# Let's debug the MCP server connection issue
from mcp.client.stdio import stdio_client
from mcp import ClientSession

# Fix the import conflict issue
import sys
print("Checking imports:")
probill_path = None
autogen_ext_path = None
for p in sys.path:
    if 'probill' in p:
        probill_path = p
        print(f"Found probill path: {p}")
    if 'autogen-ext' in p:
        autogen_ext_path = p
        print(f"Found autogen-ext path: {p}")

# Direct implementation that doesn't use the conflicting function
async def connect_to_mcp_server():
    print("Connecting to MCP server using stdio_client...")
    async with stdio_client(server_params) as (read, write):
        print("stdio_client connected successfully")
        async with ClientSession(read, write) as session:
            print("Session created successfully")
            # Initialize the connection
            await session.initialize()
            print("Session initialized successfully")
            # List available tools
            tools_list = await session.list_tools()
            print(f"Found {len(tools_list.tools)} tools")
            return tools_list.tools

# Use our custom function instead
tools = await connect_to_mcp_server()

In [ ]:
res = await mcp_host_agent.run(task="Hi")

# Let's debug the MCP server connection issue
from mcp.client.stdio import stdio_client
from mcp import ClientSession
import inspect

# Fix the import conflict issue
import sys
print("Checking imports:")
probill_path = None
autogen_ext_path = None
for p in sys.path:
    if 'probill' in p:
        probill_path = p
        print(f"Found probill path: {p}")
    if 'autogen-ext' in p:
        autogen_ext_path = p
        print(f"Found autogen-ext path: {p}")

print("\nChecking server_params type:")
print(f"Type: {type(server_params)}")
print(f"Attributes: {dir(server_params)}")

# Inspect check_and_create_server_params function
from probill.utils import check_and_create_server_params
print("\nCheck server_params utility:")
print(inspect.getsource(check_and_create_server_params))

# Direct implementation that doesn't use the conflicting function
async def connect_to_mcp_server():
    print("Connecting to MCP server using stdio_client...")
    async with stdio_client(server_params) as (read, write):
        print("stdio_client connected successfully")
        async with ClientSession(read, write) as session:
            print("Session created successfully")
            # Initialize the connection
            await session.initialize()
            print("Session initialized successfully")
            # List available tools
            tools_list = await session.list_tools()
            print(f"Found {len(tools_list.tools)} tools")
            return tools_list.tools

# Use our custom function instead
tools = await connect_to_mcp_server()

In [ ]:
for msg in res.messages:
    print("-"*10,f"{msg.type} ({msg.source})", "-"*10)
    if msg.source == "Assistant":
        if msg.type != "ThoughtEvent":
            print(json.loads(msg.content))
        else:
            print(msg.content)
    else:
        print(msg.content)


In [ ]:
agent_json= mcp_host_agent.dump_component()

In [ ]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            print("session initiated", flush=True)
            # List available prompts
            # prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            # resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
                print("*"*50)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

In [ ]:
# Create a new McpHostAgent directly
agent = McpHostAgent(
    name="obsidian_agent",
    model_client=client,
    description="Agent interacts with Obsidian contents",
    server_params=server_params,  # This should already have the type field
    model_client_stream=True,
    system_message="You are an agent uses obsidian tools to interact with obsidian contents. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

# Verify the server_params has a type field
print("Server params type:", type(server_params))
if hasattr(server_params, "__dict__"):
    print("Server params attributes:", server_params.__dict__)

In [ ]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime
from autogen_core import try_get_known_serializers_for_type, TopicId, AgentId, TypeSubscription
from autogen_agentchat.messages import TextMessage,  MultiModalMessage, StopMessage, ToolCallSummaryMessage, HandoffMessage
from probill.agents import RoutedAssistantAgent, RoutedAssistantAgentConfig
agent_name = "ObsidianAgent"

agent = RoutedAssistantAgent(
        agent_name,
        model_client=client,
        description="Agent interacts with Obsidian contents",
        system_message="Agent interacts with Obsidian contents",
        reflect_on_tool_use=True,
    )  

In [ ]:
task = """Hi!"""
async for chuck in agent.on_messages_stream(messages=[TextMessage(content="Hi!", source="user")]):
    print(chuck)
    if isinstance(chuck, TextMessage):
        print(chuck.content)
    elif isinstance(chuck, ToolCallSummaryMessage):
        print(chuck.tool_call_summary)
    elif isinstance(chuck, StopMessage):
        print("stop")
    elif isinstance(chuck, HandoffMessage):
        print("handoff")
    elif isinstance(chuck, MultiModalMessage):
        print("multi-modal")
    else:
        print("unknown message type")

In [ ]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime
from autogen_core import try_get_known_serializers_for_type, TopicId, AgentId, TypeSubscription
from autogen_agentchat.messages import TextMessage,  MultiModalMessage, StopMessage, ToolCallSummaryMessage, HandoffMessage
from probill.agents import RoutedAssistantAgent, RoutedAssistantAgentConfig
# from probill.utils import get_serializers

runtime = GrpcWorkerAgentRuntime(host_address="localhost:50051")
for msg_type in [TextMessage,  MultiModalMessage, StopMessage, ToolCallSummaryMessage, HandoffMessage]:
    runtime.add_message_serializer(try_get_known_serializers_for_type(msg_type))
await runtime.start()
agent_name = "ObsidianAgent"
agent_type = await RoutedAssistantAgent.register(
    runtime, 
    agent_name,
    lambda: RoutedAssistantAgent(
        agent_name,
        model_client=client,
        description="Agent interacts with Obsidian contents",
        system_message="Agent interacts with Obsidian contents",
        reflect_on_tool_use=True,
    )    
)
await runtime.add_subscription(
        TypeSubscription(topic_type=agent_name, agent_type=agent_type.type)
    )

await runtime.stop_when_signal()
await client.close()

In [ ]:
task = """List documents"""
result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
content = """Patient ID:
P010
Evaluation Summary:
The evaluation started at page BINV-20, which covers the systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease.

Decision Points and Therapies:
Bone Disease Presence:

Question: Does the patient have bone disease?
Answer: Condition(s) matched.
Therapy Option: Bone Disease Therapy
Regimen: Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)
Outcome: In addition to systemic therapy if bone metastases are present, expected survival ≥ 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw.
ER- and/or PR-positive and HER2-negative Status:

Question: Is the patient ER- and/or PR-positive and HER2-negative?
Answer: Condition(s) matched.
The evaluation then moved to page BINV-21, which covers systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease in patients who are ER- and/or PR-positive and HER2-negative.
Visceral Crisis:

Question: Does the patient have a visceral crisis?
Answer: Patient data missed.
This decision point was not answered due to missing data on whether the patient has a visceral crisis.
Missing Data:
VisceralCrisis
Therapies Recommended:
Bone Disease Therapy:
Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)
Expected survival ≥ 3 months
Adequate renal function required
Dental exam and monitoring for osteonecrosis of the jaw
Next Steps:
The evaluation suggests that further information is needed regarding whether the patient has a visceral crisis to determine the next course of action.
Would you like to provide additional data on the presence of a visceral crisis or need further clarification on any part of the evaluation?"""

In [ ]:
json_canvers_spec = """JSON Canvas Spec
Version 1.0 — 2024-03-11

Top level
The top level of JSON Canvas contains two arrays:

nodes (optional, array of nodes)
edges (optional, array of edges)
Nodes
Nodes are objects within the canvas. Nodes may be text, files, links, or groups.

Nodes are placed in the array in ascending order by z-index. The first node in the array should be displayed below all other nodes, and the last node in the array should be displayed on top of all other nodes.

Generic node
All nodes include the following attributes:

id (required, string) is a unique ID for the node.
type (required, string) is the node type.
text
file
link
group
x (required, integer) is the x position of the node in pixels.
y (required, integer) is the y position of the node in pixels.
width (required, integer) is the width of the node in pixels.
height (required, integer) is the height of the node in pixels.
color (optional, canvasColor) is the color of the node, see the Color section.
Text type nodes
Text type nodes store text. Along with generic node attributes, text nodes include the following attribute:

text (required, string) in plain text with Markdown syntax.
File type nodes
File type nodes reference other files or attachments, such as images, videos, etc. Along with generic node attributes, file nodes include the following attributes:

file (required, string) is the path to the file within the system.
subpath (optional, string) is a subpath that may link to a heading or a block. Always starts with a #.
Link type nodes
Link type nodes reference a URL. Along with generic node attributes, link nodes include the following attribute:

url (required, string)
Group type nodes
Group type nodes are used as a visual container for nodes within it. Along with generic node attributes, group nodes include the following attributes:

label (optional, string) is a text label for the group.
background (optional, string) is the path to the background image.
backgroundStyle (optional, string) is the rendering style of the background image. Valid values:
cover fills the entire width and height of the node.
ratio maintains the aspect ratio of the background image.
repeat repeats the image as a pattern in both x/y directions.
Edges
Edges are lines that connect one node to another.

id (required, string) is a unique ID for the edge.
fromNode (required, string) is the node id where the connection starts.
fromSide (optional, string) is the side where this edge starts. Valid values:
top
right
bottom
left
fromEnd (optional, string) is the shape of the endpoint at the edge start. Defaults to none if not specified. Valid values:
none
arrow
toNode (required, string) is the node id where the connection ends.
toSide (optional, string) is the side where this edge ends. Valid values:
top
right
bottom
left
toEnd (optional, string) is the shape of the endpoint at the edge end. Defaults to arrow if not specified. Valid values:
none
arrow
color (optional, canvasColor) is the color of the line, see the Color section.
label (optional, string) is a text label for the edge.
Color
The canvasColor type is used to encode color data for nodes and edges. Colors attributes expect a string. Colors can be specified in hex format e.g. "#FF0000", or using one of the preset colors, e.g. "1" for red. Six preset colors exist, mapped to the following numbers:

"1" red
"2" orange
"3" yellow
"4" green
"5" cyan
"6" purple
Specific values for the preset colors are intentionally not defined so that applications can tailor the presets to their specific brand colors or color scheme."""

In [ ]:
task = f"""Follow the Canvas JSON Spec: {json_canvers_spec} to convert the content below to a Canvas JSON object:\n{content}\nReturn a valid JSON object only"""
result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
canvas_json_object = {
  "nodes": [
    {
      "id": "1",
      "type": "text",
      "x": 50,
      "y": 50,
      "width": 300,
      "height": 100,
      "color": "#4",
      "text": "**Patient ID:**\nP010\n**Evaluation Summary:**\nThe evaluation started at page BINV-20, which covers the systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease."
    },
    {
      "id": "2",
      "type": "text",
      "x": 50,
      "y": 200,
      "width": 300,
      "height": 150,
      "color": "#4",
      "text": "**Decision Points and Therapies:**\n**Bone Disease Presence:**\n\nQuestion: Does the patient have bone disease?\nAnswer: Condition(s) matched.\nTherapy Option: Bone Disease Therapy\nRegimen: Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)\nOutcome: In addition to systemic therapy if bone metastases are present, expected survival ≥ 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw."
    },
    {
      "id": "3",
      "type": "text",
      "x": 50,
      "y": 400,
      "width": 300,
      "height": 150,
      "color": "#4",
      "text": "**ER- and/or PR-positive and HER2-negative Status:**\n\nQuestion: Is the patient ER- and/or PR-positive and HER2-negative?\nAnswer: Condition(s) matched.\nThe evaluation then moved to page BINV-21, which covers systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease in patients who are ER- and/or PR-positive and HER2-negative."
    },
    {
      "id": "4",
      "type": "text",
      "x": 50,
      "y": 600,
      "width": 300,
      "height": 100,
      "color": "#4",
      "text": "**Visceral Crisis:**\n\nQuestion: Does the patient have a visceral crisis?\nAnswer: Patient data missed.\nThis decision point was not answered due to missing data on whether the patient has a visceral crisis."
    },
    {
      "id": "5",
      "type": "text",
      "x": 400,
      "y": 200,
      "width": 300,
      "height": 100,
      "color": "#6",
      "text": "**Missing Data:**\nVisceralCrisis"
    },
    {
      "id": "6",
      "type": "text",
      "x": 400,
      "y": 400,
      "width": 300,
      "height": 150,
      "color": "#6",
      "text": "**Therapies Recommended:**\n**Bone Disease Therapy:**\nDenosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)\nExpected survival ≥ 3 months\nAdequate renal function required\nDental exam and monitoring for osteonecrosis of the jaw"
    },
    {
      "id": "7",
      "type": "text",
      "x": 400,
      "y": 600,
      "width": 300,
      "height": 100,
      "color": "#6",
      "text": "**Next Steps:**\nThe evaluation suggests that further information is needed regarding whether the patient has a visceral crisis to determine the next course of action.\nWould you like to provide additional data on the presence of a visceral crisis or need further clarification on any part of the evaluation?"
    }
  ],
  "edges": [
    {
      "id": "e1",
      "fromNode": "1",
      "toNode": "2",
      "color": "#4"
    },
    {
      "id": "e2",
      "fromNode": "2",
      "toNode": "3",
      "color": "#4"
    },
    {
      "id": "e3",
      "fromNode": "3",
      "toNode": "4",
      "color": "#4"
    },
    {
      "id": "e4",
      "fromNode": "2",
      "toNode": "5",
      "color": "#6"
    },
    {
      "id": "e5",
      "fromNode": "3",
      "toNode": "5",
      "color": "#6"
    },
    {
      "id": "e6",
      "fromNode": "5",
      "toNode": "7",
      "color": "#6"
    },
    {
      "id": "e7",
      "fromNode": "4",
      "toNode": "7",
      "color": "#6"
    }
  ]
}

In [ ]:
task = f"""Append the given JSON object as the content to file: 'patient_data.canvas':\nJSON object:\n{json.dumps(canvas_json_object)}"""
agent.on_reset
result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
task = "List your tools and delete tools.md, then, append the tool list to it"
agent.on_reset
result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
agent_json = export_component(agent)
print(agent_json.model_dump_json(indent=2))